# Model: "value_parametric" (for vmPFC value encoding analysis)

Tests whether brain activity (especially vmPFC) correlates with stimulus value across all trials in session 1 (week 1), when stimuli are still relatively novel.

Design matrix regressors:
  - `stim_ev`: stimulus onset (unmodulated), all trials  
  - `stim_value_par`: parametric modulation by stimulus value at stimulus onset  
  - `feedback_ev`: feedback onset (unmodulated), all trials  
  - `reward_par`: parametric modulation by reward at feedback onset  
  - ~~`choice_st`: stick function at time of choice (stimulus offset / feedback onset)~~
      - collinear with `feedback_ev`
      - if you want to model the motor response explicitly don't include `feedback_ev`
  - `cross_ev`: fixation cross events  
  - `confounds`: motion parameters, framewise displacement, dvars, scrub regressors   

For the yesNo task:
  - `stim_value_par` uses `valStim_dmn` (the objective stimulus value, demeaned)
  
For the binaryChoice task:
  - `stim_value_par` uses `valChosenMinusUnchosen_dmn` (value difference at stimulus onset)
  
**NOTE**: This model intentionally does NOT split by HT vs RE. The goal is to test overall value encoding across all trials, before examining HT/RE differences.

## Why the `stim_value_par` regressors are demeaned

The parametric modulator values get multiplied with the event timecourse before HRF convolution, so demeaning ensures the parametric regressor is orthogonal to its corresponding unmodulated event regressor. Without demeaning, the two would be highly collinear because the parametric regressor would contain a large component that just looks like the unmodulated one.  

Say you have 3 trials with stimulus values of 10, 20, 30. The mean is 20.
Without demeaning, the parametric regressor's timecourse (before HRF convolution) looks like:

```
stim_ev:         1  at trial onsets,  0  elsewhere
stim_value_par:  10, 20, 30  at trial onsets,  0  elsewhere
```

That 10, 20, 30 signal can be decomposed into 20, 20, 20 (a scaled copy of stim_ev) plus -10, 0, 10 (the actual variation around the mean). So most of the parametric regressor is just recapitulating the unmodulated one, scaled by the mean value. After HRF convolution, the two end up highly correlated because convolution is linear: convolving 10, 20, 30 with the HRF gives you 20 * HRF(stim_ev) + HRF(-10, 0, 10).

With demeaning, you subtract the mean (20) first, so the values become -10, 0, 10:

```
stim_ev:         1, 1, 1  at trial onsets
stim_value_par:  -10, 0, 10  at trial onsets
```

Now the parametric regressor only captures how value deviates from the average trial. It has positive and negative values that sum to zero, so it's no longer carrying a component that looks like "a stimulus appeared." The two regressors capture genuinely different things: stim_ev captures the average response to any stimulus, and stim_value_par captures how the response scales with value above or below the mean.

Standardizing (dividing by the standard deviation on top of demeaning) doesn't affect the statistical maps. The t-statistics and p-values are scale-invariant because the GLM estimates both the beta and its standard error, so multiplying the regressor by a constant just scales the beta by 1/constant and the standard error by the same factor, leaving the t-value unchanged. The only thing that changes is the units of the beta: with raw demeaned values, a beta of 0.5 means "0.5 units of BOLD change per unit of stimulus value"; with standardized values, the same beta would mean "per standard deviation of stimulus value."

## Why the `stim_value_par` regressors are *not* standardized

If you want to be able to look at the betas for a voxel and be able to answer a question like “is the effect of stimulus presentation larger than the effect of stimulus value?” you would need to standardize all the columns in the design matrix. Without standardizing we could answer questions like this by comparing the t-values.

The t-values are already comparable across regressors because they're self-normalizing (beta divided by its own standard error), so you can compare t-values for `stim_ev` and `stim_value_par` directly to ask which effect is more reliably detected at a given voxel.  

But if you want to compare the betas themselves, you run into a units problem. The beta for `stim_ev` is in units of BOLD change per unit of modulation, and since modulation is 1 for every trial, it's just "BOLD change when a stimulus appears." The beta for `stim_value_par` is in units of BOLD change per unit of `valStim_dmn`, which is in points. Those are different scales, so the raw beta magnitudes aren't directly comparable.  

Standardizing the columns (z-scoring the convolved regressors in the design matrix so they all have mean 0 and standard deviation 1) would put all betas into the same units: BOLD change per standard deviation of that regressor. Then you could meaningfully compare beta magnitudes across regressors.  

That said, there's a subtlety worth keeping in mind. For the unmodulated regressors like `stim_ev`, standardizing changes what the beta represents in a less intuitive way. The original beta tells you "how much BOLD increases when a stimulus is on screen." The standardized beta tells you "how much BOLD increases per standard deviation of the stimulus-presence timecourse," which is harder to interpret on its own.   

Standardizing is most useful when comparing effect sizes is the explicit goal, and for that purpose comparing t-values is often the more straightforward approach since it doesn't require changing the model.

## Where is RT in this model

### Option 1: RT as duration of `stim_ev` and `stim_value_par`

This is what we described above so far. Because the stimulus duration in the events file is the time from stimulus onset to the button press (i.e., the reaction time), and both `stim_ev` and `stim_value_par` use that duration. So trials where the participant responded quickly produce shorter boxcars and trials where they deliberated longer produce wider boxcars, all before HRF convolution.  

This means those regressors are capturing a mix of two things: the stimulus/value-related neural response and RT-related variance (which could reflect decision difficulty, attention, uncertainty, etc.). For `stim_value_par` specifically, this is worth thinking about because RT and stimulus value are likely correlated (e.g., participants may respond faster to high-value or very-low-value stimuli where the decision is easy). So some of the "value" signal could actually be RT-driven, and some RT variance gets absorbed into what looks like a value effect.

### Option 2: Fixed duration for stimulus regressors [BAD?]

You set the duration for `stim_ev` and `stim_value_par` to some constant, say 0 (stick function) or the median RT across trials. Every trial gets the same shaped boxcar regardless of how long the participant took. This removes RT from the stimulus regressors entirely. The design matrix columns would be the same as what we have now, but the underlying timecourses would look different because all trials produce identical temporal profiles before being scaled by the parametric modulator. The cost is that you're mismodeling the actual neural event duration, and the unmodeled RT variance ends up in the residuals, potentially reducing sensitivity.

### Option 3: Keep variable duration but add RT as a parametric modulator 

You keep the stimulus duration as the actual RT (which is what we have now), but add a new regressor that explicitly models RT-related variance at stimulus onset. The design matrix would then have:

```
cross_ev          fixation cross, unmodulated
stim_ev           stimulus onset, duration = RT, unmodulated
stim_value_par    stimulus onset, duration = RT, modulated by demeaned value
stim_rt_par       stimulus onset, duration = RT, modulated by demeaned RT
feedback_ev       feedback onset, unmodulated
reward_par        feedback onset, modulated by demeaned reward
```

The idea is that `stim_rt_par` soaks up variance related to how long the participant deliberated, so whatever remains in `stim_value_par` reflects value encoding above and beyond what can be explained by decision time. This is the more common approach in the decision neuroscience literature because it lets the model account for both sources of variance rather than discarding one.   

There's a practical concern though. RT determines the duration of the boxcar for all three stimulus regressors, and `stim_rt_par` is also modulated by RT, so you'd want to check whether this introduces problematic collinearity. In practice it's usually tolerable because the duration-based RT effect (wider boxcar) and the modulation-based RT effect (scaled amplitude) produce different timecourse shapes after HRF convolution, but it's something to verify in the VIF diagnostics.

### Option 4: Fixed stim duration + RT modulator

This gives the cleanest separation because RT only enters the model once, as an amplitude modulation, rather than through both duration and modulation. But again, you're making an assumption that the neural event has a fixed duration regardless of actual deliberation time.

## Generate reports and save design matrices for all subjects, ses-01

In [ ]:
# %% Configuration

import os
import sys

BIDS_PATH = '/hopper/groups/enkavilab/data/novel-vs-repeated/fmri/bids'
OUTPUT_DIR = '/hopper/groups/enkavilab/users/zenkavi/novel_vs_repeated/outputs/glm'

SUBJECTS = ['601', '609', '611', '619', '621', '629']
SESSION = '01'
MNUM = 'value_parametric'
SPACE = 'MNI152NLin2009cAsym_res-2'
HRF_MODEL = 'spm'
DRIFT_MODEL = 'cosine'
SCRUB_THRESH = 0.5

MODEL_VARIANTS = [
    'rt_in_duration',
    'fixed_duration',
    'rt_duration_plus_mod',
    'fixed_duration_plus_mod',
]

# %% Generate reports and design matrices for all subjects x all variants
#
# Output structure:
#   {OUTPUT_DIR}/{MNUM}/{variant}/sub-{subnum}/ses-{session}/
#       sub-{subnum}_ses-{session}_task-*_design_matrix.csv
#       sub-{subnum}_ses-{session}_*_contrasts.json
#       sub-{subnum}_ses-{session}_*_report.html

from level1_helpers import generate_report

for variant in MODEL_VARIANTS:
    for subnum in SUBJECTS:
        print(f"\n{'='*60}")
        print(f"sub-{subnum} ses-{SESSION} | {variant}")
        print(f"{'='*60}")

        try:
            report_path, design_matrices = generate_report(
                subnum=subnum,
                session=SESSION,
                data_path=BIDS_PATH,
                model_variant=variant,
                mnum=MNUM,
                space=SPACE,
                hrf_model=HRF_MODEL,
                drift_model=DRIFT_MODEL,
                scrub_thresh=SCRUB_THRESH,
                output_dir=OUTPUT_DIR,
            )
            print(f"  Saved {len(design_matrices)} design matrices and report.")
        except Exception as e:
            print(f"  FAILED: {e}")
            import traceback
            traceback.print_exc()

# Run level 1 GLMs

In [ ]:
# %% Configuration

import os
import sys

BIDS_PATH = '/hopper/groups/enkavilab/data/novel-vs-repeated/fmri/bids'
OUTPUT_DIR = '/hopper/groups/enkavilab/users/zenkavi/novel_vs_repeated/outputs/glm'

SUBJECTS = ['601', '609', '611', '619', '621', '629']
SESSION = '01'
MNUM = 'value_parametric'
SPACE = 'MNI152NLin2009cAsym_res-2'
TASKS = ['yesNo', 'binaryChoice']

# Variants to fit
MODEL_VARIANTS = [
    'rt_in_duration',
    'rt_duration_plus_mod',
]

# GLM parameters (should match what was used to build design matrices)
HRF_MODEL = 'spm'
DRIFT_MODEL = 'cosine'
NOISE_MODEL = 'ar1'
SMOOTHING_FWHM = 5

# %% Run level 1 GLMs for all subjects x tasks x variants
#
# For each combination, this will:
#   1. Load pre-saved design matrices from:
#      {OUTPUT_DIR}/{MNUM}/{variant}/sub-{subnum}/ses-{session}/
#   2. Load preprocessed BOLD images from BIDS derivatives
#   3. Fit the GLM (fixed effects across runs for yesNo; single run for binaryChoice)
#   4. Save GLM pickle, contrast maps, and contrasts JSON to:
#      {OUTPUT_DIR}/{MNUM}/{variant}/sub-{subnum}/ses-{session}/contrasts/

from level1_helpers import run_level1_pipeline

for variant in MODEL_VARIANTS:
    for task in TASKS:
        for subnum in SUBJECTS:
            print(f"\n{'='*60}")
            print(f"sub-{subnum} ses-{SESSION} task-{task} | {variant}")
            print(f"{'='*60}")

            try:
                run_level1_pipeline(
                    subnum=subnum,
                    session=SESSION,
                    task=task,
                    data_path=BIDS_PATH,
                    dm_dir=OUTPUT_DIR,
                    output_dir=OUTPUT_DIR,
                    mnum=MNUM,
                    model_variant=variant,
                    space=SPACE,
                    noise_model=NOISE_MODEL,
                    hrf_model=HRF_MODEL,
                    drift_model=DRIFT_MODEL,
                    smoothing_fwhm=SMOOTHING_FWHM,
                )
            except Exception as e:
                print(f"  FAILED: {e}")
                import traceback
                traceback.print_exc()

# Run group level GLMs

## Uncorrected maps

In [ ]:
# %% Configuration
 
import os
import sys
sys.path.insert(0, '.')
 
OUTPUT_DIR = '/hopper/groups/enkavilab/users/zenkavi/novel_vs_repeated/outputs/glm'
 
SUBJECTS = ['601', '609', '611', '619', '621', '629']
SESSION = '01'
MNUM = 'value_parametric'
SPACE = 'MNI152NLin2009cAsym_res-2'
 
CONTRAST_ID = 'stim_value_par'
 
MODEL_VARIANTS = [
    'rt_in_duration',
    'rt_duration_plus_mod',
]
 
TASKS = ['yesNo', 'binaryChoice']
 
# Uncorrected report parameters
T_THRESHOLD = 3.0
CLUSTER_THRESHOLD = 10

In [ ]:
from level2_helpers import run_group_onesample
 
for variant in MODEL_VARIANTS:
    for task in TASKS:
        print(f"\n{'='*60}")
        print(f"Fitting parametric: task-{task} {CONTRAST_ID} | {variant}")
        print(f"{'='*60}")
 
        try:
            run_group_onesample(
                subjects=SUBJECTS,
                session=SESSION,
                task=task,
                contrast_id=CONTRAST_ID,
                output_dir=OUTPUT_DIR,
                mnum=MNUM,
                model_variant=variant,
                space=SPACE,
            )
        except Exception as e:
            print(f"  FAILED: {e}")
            import traceback
            traceback.print_exc()

### Generate group reports

In [1]:
# %% Step 2: Generate reports from saved maps
#
# Reads the group tmaps saved in step 1 and generates HTML reports.
# This can be re-run independently (e.g., with different thresholds)
# without re-fitting the models.
#
# Output:
#   {OUTPUT_DIR}/{MNUM}/{variant}/group/ses-{session}/
#       group_task-{task}_..._stim_value_par_uncorrected_report.html

import os
import sys
sys.path.insert(0, '.')
 
OUTPUT_DIR = '/hopper/groups/enkavilab/users/zenkavi/novel_vs_repeated/outputs/glm'
 
SUBJECTS = ['601', '609', '611', '619', '621', '629']
SESSION = '01'
MNUM = 'value_parametric'
SPACE = 'MNI152NLin2009cAsym_res-2'
 
CONTRAST_ID = 'stim_value_par'
 
MODEL_VARIANTS = [
    'rt_in_duration',
    'rt_duration_plus_mod',
]
 
TASKS = ['yesNo', 'binaryChoice']
 
# Uncorrected report parameters
T_THRESHOLD = 3.0
CLUSTER_THRESHOLD = 10

from level2_report_uncorrected import generate_group_report
 
for variant in MODEL_VARIANTS:
    for task in TASKS:
        print(f"\n{'='*60}")
        print(f"Uncorrected report: task-{task} {CONTRAST_ID} | {variant}")
        print(f"{'='*60}")
 
        try:
            generate_group_report(
                subjects=SUBJECTS,
                session=SESSION,
                task=task,
                contrast_id=CONTRAST_ID,
                output_dir=OUTPUT_DIR,
                mnum=MNUM,
                model_variant=variant,
                space=SPACE,
                threshold=T_THRESHOLD,
                cluster_threshold=CLUSTER_THRESHOLD,
            )
        except Exception as e:
            print(f"  FAILED: {e}")
            import traceback
            traceback.print_exc()


Uncorrected report: task-yesNo stim_value_par | rt_in_duration
  Generating report: task-yesNo stim_value_par [rt_in_duration] (n=6)


/hopper/groups/enkavilab/pyenvs/fmri/lib64/python3.9/site-packages/numpy/_core/fromnumeric.py:820: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)


  Report saved: /hopper/groups/enkavilab/users/zenkavi/novel_vs_repeated/outputs/glm/value_parametric/rt_in_duration/group/ses-01/group_task-yesNo_space-MNI152NLin2009cAsym_res-2_value_parametric_rt_in_duration_stim_value_par_uncorrected_report.html

Uncorrected report: task-binaryChoice stim_value_par | rt_in_duration
  Generating report: task-binaryChoice stim_value_par [rt_in_duration] (n=6)


/hopper/groups/enkavilab/pyenvs/fmri/lib64/python3.9/site-packages/numpy/_core/fromnumeric.py:820: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)


  Report saved: /hopper/groups/enkavilab/users/zenkavi/novel_vs_repeated/outputs/glm/value_parametric/rt_in_duration/group/ses-01/group_task-binaryChoice_space-MNI152NLin2009cAsym_res-2_value_parametric_rt_in_duration_stim_value_par_uncorrected_report.html

Uncorrected report: task-yesNo stim_value_par | rt_duration_plus_mod
  Generating report: task-yesNo stim_value_par [rt_duration_plus_mod] (n=6)


/hopper/groups/enkavilab/pyenvs/fmri/lib64/python3.9/site-packages/numpy/_core/fromnumeric.py:820: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)


  Report saved: /hopper/groups/enkavilab/users/zenkavi/novel_vs_repeated/outputs/glm/value_parametric/rt_duration_plus_mod/group/ses-01/group_task-yesNo_space-MNI152NLin2009cAsym_res-2_value_parametric_rt_duration_plus_mod_stim_value_par_uncorrected_report.html

Uncorrected report: task-binaryChoice stim_value_par | rt_duration_plus_mod
  Generating report: task-binaryChoice stim_value_par [rt_duration_plus_mod] (n=6)


/hopper/groups/enkavilab/pyenvs/fmri/lib64/python3.9/site-packages/numpy/_core/fromnumeric.py:820: UserWarning: Warning: 'partition' will ignore the 'mask' of the MaskedArray.
  a.partition(kth, axis=axis, kind=kind, order=order)


  Report saved: /hopper/groups/enkavilab/users/zenkavi/novel_vs_repeated/outputs/glm/value_parametric/rt_duration_plus_mod/group/ses-01/group_task-binaryChoice_space-MNI152NLin2009cAsym_res-2_value_parametric_rt_duration_plus_mod_stim_value_par_uncorrected_report.html


### Generate rt comparison report

In [2]:
# %% Configuration

import os
import sys
sys.path.insert(0, '.')

OUTPUT_DIR = '/hopper/groups/enkavilab/users/zenkavi/novel_vs_repeated/outputs/glm'

SUBJECTS = ['601', '609', '611', '619', '621', '629']
SESSION = '01'
MNUM = 'value_parametric'
SPACE = 'MNI152NLin2009cAsym_res-2'
CONTRAST_ID = 'stim_value_par'

MODEL_VARIANTS = [
    'rt_in_duration',
    'rt_duration_plus_mod',
]

TASKS = ['yesNo', 'binaryChoice']

T_THRESHOLD = 3.0

# %% Generate comparison report

from level2_report_comparison import generate_comparison_report

print(f"\n{'='*60}")
print(f"Comparison report: {CONTRAST_ID}")
print(f"{'='*60}")

try:
    generate_comparison_report(
        subjects=SUBJECTS,
        session=SESSION,
        contrast_id=CONTRAST_ID,
        output_dir=OUTPUT_DIR,
        tasks=TASKS,
        model_variants=MODEL_VARIANTS,
        mnum=MNUM,
        space=SPACE,
        threshold=T_THRESHOLD,
    )
except Exception as e:
    print(f"  FAILED: {e}")
    import traceback
    traceback.print_exc()


Comparison report: stim_value_par
  Loaded: rt_in_duration / yesNo
  Loaded: rt_in_duration / binaryChoice
  Loaded: rt_duration_plus_mod / yesNo
  Loaded: rt_duration_plus_mod / binaryChoice
  Comparison report saved: /hopper/groups/enkavilab/users/zenkavi/novel_vs_repeated/outputs/glm/value_parametric/rt_effect_comparison/group/ses-01/comparison_stim_value_par_rt_in_duration_vs_rt_duration_plus_mod_uncorrected_report.html


## Non-parametric inference

With only 6 subjects, it's very unlikely anything would surivive TFCE. TFCE with permutation testing is a very conservative correction, and with n=6, the permutation distribution of the maximum TFCE statistic is estimated from only 2^6 = 64 possible sign-flip permutations (since the one-sample test uses sign-flipping rather than full permutation). That's a very coarse distribution to estimate family-wise error rates from, so even genuinely large effects can fail to reach significance.

The sign-flipping permutation test for a one-sample design works by randomly flipping the sign of each subject's map (multiplying by +1 or -1) on each permutation. With 6 subjects, each subject can be flipped or not, giving 2^6 = 64 unique sign-flip combinations. One of those 64 is the original (unflipped) data.
The FWE-corrected p-value for a voxel is defined as: "out of all 64 permutations, what fraction produced a maximum test statistic (across the whole brain) at least as large as the one I observed?" The smallest possible p-value occurs when the observed statistic is the most extreme out of all 64 permutations, meaning zero permutations exceeded it. In that case, the p-value is 1/64 = 0.016 (nilearn includes the observed data as one of the permutations, so the minimum is 1/64, not 0/64).  

That floor of 0.016 means any individual voxel can in principle reach significance at alpha = 0.05. So my earlier statement was slightly misleading. What I should have said is that the resolution of the p-value distribution is very coarse: your possible p-values are 1/64, 2/64, 3/64, and so on. You can reach p = 0.016 at best, and the next step is p = 0.031, then p = 0.047 (which is just barely under 0.05). So for a voxel to survive at alpha = 0.05, the observed maximum TFCE value needs to be in the top 3 out of 64 permutations.  

The practical problem is that TFCE produces a single maximum TFCE value per permutation across the entire brain. With ~200,000 voxels and highly variable noise from only 6 subjects, even permutations with flipped signs can produce large maximum TFCE values somewhere in the brain by chance. So the permutation distribution of the maximum TFCE statistic ends up quite high, and your observed values at vmPFC, even though they have large t-statistics, may not exceed enough of those 64 permutation maxima to land in the top 3.  

In [ ]:
# %% Step 1b: Run non-parametric inference with TFCE
#
# This is the slow step. With 10000 permutations and 6 subjects,
# expect ~10-30 min per task x variant combination depending on
# image size and n_jobs.
#
# Output:
#   {OUTPUT_DIR}/{MNUM}/{variant}/group/ses-{session}/
#       ..._stim_value_par_tfce_tmap.nii.gz
#       ..._stim_value_par_tfce_logp.nii.gz

from datetime import datetime
import os
import sys
sys.path.insert(0, '.')
 
OUTPUT_DIR = '/hopper/groups/enkavilab/users/zenkavi/novel_vs_repeated/outputs/glm'
 
SUBJECTS = ['601', '609', '611', '619', '621', '629']
SESSION = '01'
MNUM = 'value_parametric'
SPACE = 'MNI152NLin2009cAsym_res-2'
 
CONTRAST_ID = 'stim_value_par'
 
MODEL_VARIANTS = [
    'rt_in_duration',
    'rt_duration_plus_mod',
]
 
TASKS = ['yesNo', 'binaryChoice']
 
# Uncorrected report parameters
T_THRESHOLD = 3.0
CLUSTER_THRESHOLD = 10
 
# Non-parametric inference parameters
N_PERM = 10000
N_JOBS = 32         # adjust based on cluster resources
ALPHA = 0.05


from level2_helpers import run_group_nonparametric
 
# for variant in MODEL_VARIANTS:
#     for task in TASKS:
#         print(f"\n{'='*60}")
#         print(f"TFCE: task-{task} {CONTRAST_ID} | {variant}")
#         from datetime import datetime
#         print(f"  Started at: {datetime.now().strftime('%H:%M:%S')}")
#         print(f"{'='*60}")
 
#         try:
#             run_group_nonparametric(
#                 subjects=SUBJECTS,
#                 session=SESSION,
#                 task=task,
#                 contrast_id=CONTRAST_ID,
#                 output_dir=OUTPUT_DIR,
#                 mnum=MNUM,
#                 model_variant=variant,
#                 space=SPACE,
#                 n_perm=N_PERM,
#                 n_jobs=N_JOBS,
#             )
#         except Exception as e:
#             print(f"  FAILED: {e}")
#             import traceback
#             traceback.print_exc()

### Generate group reports

In [4]:
# %% Step 2b: Generate TFCE-corrected reports

from datetime import datetime
import os
import sys
sys.path.insert(0, '.')
 
OUTPUT_DIR = '/hopper/groups/enkavilab/users/zenkavi/novel_vs_repeated/outputs/glm'
 
SUBJECTS = ['601', '609', '611', '619', '621', '629']
SESSION = '01'
MNUM = 'value_parametric'
SPACE = 'MNI152NLin2009cAsym_res-2'
 
CONTRAST_ID = 'stim_value_par'
 
MODEL_VARIANTS = [
    'rt_in_duration',
    'rt_duration_plus_mod',
]
 
TASKS = ['yesNo', 'binaryChoice']
 
# Uncorrected report parameters
T_THRESHOLD = 3.0
CLUSTER_THRESHOLD = 10
 
# Non-parametric inference parameters
N_PERM = 10000
N_JOBS = 32         # adjust based on cluster resources
ALPHA = 0.05


from level2_report_corrected import generate_group_report_corrected
 
for variant in MODEL_VARIANTS:
    for task in TASKS:
        print(f"\n{'='*60}")
        print(f"TFCE report: task-{task} {CONTRAST_ID} | {variant}")
        print(f"{'='*60}")
 
        try:
            generate_group_report_corrected(
                subjects=SUBJECTS,
                session=SESSION,
                task=task,
                contrast_id=CONTRAST_ID,
                output_dir=OUTPUT_DIR,
                mnum=MNUM,
                model_variant=variant,
                space=SPACE,
                alpha=ALPHA,
            )
        except Exception as e:
            print(f"  FAILED: {e}")
            import traceback
            traceback.print_exc()


TFCE report: task-yesNo stim_value_par | rt_in_duration
  Generating TFCE report: task-yesNo stim_value_par [rt_in_duration] (n=6, 0 significant voxels at alpha=0.05)


/hopper/groups/enkavilab/users/zenkavi/novel_vs_repeated/src/glm/./level2_helpers.py:423: UserWarning: empty mask
  plot_glass_brain(
/hopper/groups/enkavilab/users/zenkavi/novel_vs_repeated/src/glm/./level2_helpers.py:435: UserWarning: empty mask
  display = plot_stat_map(


  Report saved: /hopper/groups/enkavilab/users/zenkavi/novel_vs_repeated/outputs/glm/value_parametric/rt_in_duration/group/ses-01/group_task-yesNo_space-MNI152NLin2009cAsym_res-2_value_parametric_rt_in_duration_stim_value_par_tfce_corrected_report.html

TFCE report: task-binaryChoice stim_value_par | rt_in_duration
  Generating TFCE report: task-binaryChoice stim_value_par [rt_in_duration] (n=6, 0 significant voxels at alpha=0.05)
  Report saved: /hopper/groups/enkavilab/users/zenkavi/novel_vs_repeated/outputs/glm/value_parametric/rt_in_duration/group/ses-01/group_task-binaryChoice_space-MNI152NLin2009cAsym_res-2_value_parametric_rt_in_duration_stim_value_par_tfce_corrected_report.html

TFCE report: task-yesNo stim_value_par | rt_duration_plus_mod
  Generating TFCE report: task-yesNo stim_value_par [rt_duration_plus_mod] (n=6, 0 significant voxels at alpha=0.05)
  Report saved: /hopper/groups/enkavilab/users/zenkavi/novel_vs_repeated/outputs/glm/value_parametric/rt_duration_plus_mod/gr

## ROI reports

In [5]:
# %% Configuration

import os
import sys
sys.path.insert(0, '.')

OUTPUT_DIR = '/hopper/groups/enkavilab/users/zenkavi/novel_vs_repeated/outputs/glm'

SUBJECTS = ['601', '609', '611', '619', '621', '629']
SESSION = '01'
MNUM = 'value_parametric'
SPACE = 'MNI152NLin2009cAsym_res-2'
CONTRAST_ID = 'stim_value_par'

MODEL_VARIANTS = [
    'rt_in_duration',
    'rt_duration_plus_mod',
]

TASKS = ['yesNo', 'binaryChoice']

T_THRESHOLD = 3.0
CLUSTER_THRESHOLD = 10
ALPHA = 0.05

# %% Generate ROI reports

from level2_report_roi import generate_roi_report

for variant in MODEL_VARIANTS:
    for task in TASKS:
        print(f"\n{'='*60}")
        print(f"ROI report: task-{task} {CONTRAST_ID} | {variant}")
        print(f"{'='*60}")

        try:
            generate_roi_report(
                subjects=SUBJECTS,
                session=SESSION,
                task=task,
                contrast_id=CONTRAST_ID,
                output_dir=OUTPUT_DIR,
                mnum=MNUM,
                model_variant=variant,
                space=SPACE,
                threshold=T_THRESHOLD,
                cluster_threshold=CLUSTER_THRESHOLD,
                alpha=ALPHA,
            )
        except Exception as e:
            print(f"  FAILED: {e}")
            import traceback
            traceback.print_exc()


ROI report: task-yesNo stim_value_par | rt_in_duration
  Generating ROI report: task-yesNo stim_value_par [rt_in_duration] (n=6)
  ROI report saved: /hopper/groups/enkavilab/users/zenkavi/novel_vs_repeated/outputs/glm/value_parametric/rt_in_duration/group/ses-01/group_task-yesNo_space-MNI152NLin2009cAsym_res-2_value_parametric_rt_in_duration_stim_value_par_roi_report.html

ROI report: task-binaryChoice stim_value_par | rt_in_duration
  Generating ROI report: task-binaryChoice stim_value_par [rt_in_duration] (n=6)


/hopper/groups/enkavilab/users/zenkavi/novel_vs_repeated/src/glm/./level2_helpers.py:435: UserWarning: empty mask
  display = plot_stat_map(


  ROI report saved: /hopper/groups/enkavilab/users/zenkavi/novel_vs_repeated/outputs/glm/value_parametric/rt_in_duration/group/ses-01/group_task-binaryChoice_space-MNI152NLin2009cAsym_res-2_value_parametric_rt_in_duration_stim_value_par_roi_report.html

ROI report: task-yesNo stim_value_par | rt_duration_plus_mod
  Generating ROI report: task-yesNo stim_value_par [rt_duration_plus_mod] (n=6)


/hopper/groups/enkavilab/users/zenkavi/novel_vs_repeated/src/glm/./level2_helpers.py:435: UserWarning: empty mask
  display = plot_stat_map(


  ROI report saved: /hopper/groups/enkavilab/users/zenkavi/novel_vs_repeated/outputs/glm/value_parametric/rt_duration_plus_mod/group/ses-01/group_task-yesNo_space-MNI152NLin2009cAsym_res-2_value_parametric_rt_duration_plus_mod_stim_value_par_roi_report.html

ROI report: task-binaryChoice stim_value_par | rt_duration_plus_mod
  Generating ROI report: task-binaryChoice stim_value_par [rt_duration_plus_mod] (n=6)


/hopper/groups/enkavilab/users/zenkavi/novel_vs_repeated/src/glm/./level2_helpers.py:435: UserWarning: empty mask
  display = plot_stat_map(


  ROI report saved: /hopper/groups/enkavilab/users/zenkavi/novel_vs_repeated/outputs/glm/value_parametric/rt_duration_plus_mod/group/ses-01/group_task-binaryChoice_space-MNI152NLin2009cAsym_res-2_value_parametric_rt_duration_plus_mod_stim_value_par_roi_report.html
